# ReceiptGraphKIE: LayoutLMv3 with Symbolic Relation-GATv2 and Word-Level CRF

This notebook trains and evaluates a graph-enhanced key information extraction model on the CORD receipt dataset. The architecture combines pretrained LayoutLMv3 representations, a two-layer relation-aware GATv2, hidden-state fusion, separate base and fusion classifiers, and a word-level CRF.

Each receipt is represented as a symbolic word graph. Nodes correspond to words, while typed edges encode exact line membership, cross-line column continuity, directional geometry, and local fallback proximity. The graph branch performs relation-aware message passing and contributes a fixed-scale residual to the LayoutLMv3 word representations before structured CRF decoding.

The evaluation protocol includes:

- Five independent runs with seeds `13`, `42`, `2026`, `7`, and `123`.
- Fresh initialization from pretrained LayoutLMv3 plus randomly initialized graph and task heads for every seed.
- Fixed graph scale `0.50`, base auxiliary weight `0.35`, and early-stopping patience `6`.
- Checkpoint selection based exclusively on development-set Entity Macro F1 in the original graph mode.
- Four development-set graph conditions: `original`, `shuffled`, `self_loop_only`, and `graph_off`.
- Test evaluation only after all five training runs and development ablations are complete.
- Mean ± sample standard deviation reporting, attention diagnostics, document-level predictions, and absolute document bootstrap intervals.

The primary metric is macro F1 over 18 semantic entity classes, excluding the background label `O`.


## 1. Kaggle Runtime Setup and Environment Validation

This section installs only missing dependencies, imports the required machine-learning and graph-processing libraries, verifies CUDA availability, locates the CORD dataset, and initializes the experiment workspace. The existing Kaggle PyTorch installation is retained to maintain CUDA compatibility.


In [1]:
import importlib.util
import subprocess
import sys

REQUIRED = {
    "torch_geometric": "torch-geometric>=2.4,<3",
    "torchcrf": "pytorch-crf>=0.7.2,<1",
    "transformers": "transformers>=4.30,<6",
    "sklearn": "scikit-learn>=1.2,<2",
}

for import_name, package in REQUIRED.items():
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("Dependencies are ready. Restart the kernel only if Kaggle requests it.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.2 MB/s eta 0:00:00
Dependencies are ready. Restart the kernel only if Kaggle requests it.


In [2]:
import gc
import hashlib
import json
import math
import os
import platform
import random
import shutil
import time
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_geometric
import transformers
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_recall_fscore_support
from scipy.stats import t as student_t
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torch_geometric.data import Batch, Data
from torch_geometric.nn import GATv2Conv
from torchcrf import CRF
from tqdm.auto import tqdm
from transformers import LayoutLMv3Model, LayoutLMv3Processor, get_linear_schedule_with_warmup

assert torch.cuda.is_available(), "Hãy bật GPU trong Kaggle trước khi train."

print("Runtime: Kaggle | Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("Transformers:", transformers.__version__)
print("PyG:", torch_geometric.__version__)
print("scikit-learn:", sklearn.__version__)


Runtime: Kaggle | Python: 3.12.13
PyTorch: 2.10.0+cu128
CUDA: 12.8
GPU: Tesla T4
Transformers: 5.0.0
PyG: 2.8.0.post1
scikit-learn: 1.6.1


## 2. Experimental Configuration and Evaluation Protocol

This section defines the architecture, optimization schedule, regularization, graph parameters, reproducibility controls, and artifact paths.

The model is trained exactly five times using seeds `13`, `42`, `2026`, `7`, and `123`. Every run uses the same fixed configuration: two Relation-GATv2 layers, graph scale `0.50`, base auxiliary weight `0.35`, and early-stopping patience `6`. No hyperparameter sweep is performed within this experiment.

Each seed starts from the pretrained `microsoft/layoutlmv3-base` checkpoint with independently initialized graph, fusion, classification, and CRF parameters. Development Entity Macro F1 in the original graph mode is the sole checkpoint-selection criterion.


In [3]:
@dataclass(frozen=True)
class Config:
    profile: str = "paper"
    model_id: str = "microsoft/layoutlmv3-base"
    model_revision: str = "main"
    max_length: int = 512
    batch_size: int = 2
    grad_accum_steps: int = 2
    num_workers: int = 2

    backbone_lr: float = 1e-5
    head_lr: float = 1e-4
    graph_lr: float = 2e-4
    weight_decay: float = 0.01
    warmup_ratio: float = 0.10
    max_epochs: int = 14
    patience: int = 6
    min_delta: float = 1e-4
    freeze_backbone_epochs: int = 1
    lm_dropout: float = 0.30
    gat_attention_dropout: float = 0.10
    graph_dropout: float = 0.10
    fusion_dropout: float = 0.10
    gat_layers: int = 2
    gat_heads: int = 4
    gat_edge_hidden: int = 64
    gat_ffn_multiplier: int = 1
    gat_residual_scale: float = 0.50
    spatial_input_scale: float = 0.10
    relation_types: int = 7
    edge_dim: int = 18
    edge_dropout: float = 0.00

    fixed_graph_alpha: float = 0.50
    base_aux_weight: float = 0.35
    graph_branch_dropout: float = 0.10
    max_graph_neighbors: int = 8
    graph_shuffle_seed: int = 1729
    column_center_threshold: float = 0.10
    column_overlap_threshold: float = 0.20

    crf_weight: float = 0.65
    focal_gamma: float = 2.0
    knn_k: int = 2
    use_rare_document_sampler: bool = True
    rare_sampler_power: float = 0.50
    rare_sampler_max_weight: float = 2.50
    class_balance_beta: float = 0.999
    class_weight_cap: float = 3.0
    use_amp: bool = True
    keep_all_checkpoints: bool = True
    keep_nonselected_alpha_checkpoints: bool = False
    bootstrap_repetitions: int = 10000
    track_epoch_graph_off: bool = True
    artifact_dir: str = "/kaggle/working/receipt_kie_five_seed_retrain_artifacts"


CFG = Config()
SEEDS = [13, 42, 2026, 7, 123]
MAX_EPOCHS = CFG.max_epochs
PATIENCE = CFG.patience
SELECTED_ALPHA = CFG.fixed_graph_alpha
SELECTED_BASE_AUX = CFG.base_aux_weight
MODEL_NAME = "layoutlmv3_split_head_symbolic_rel_gatv2_fusion_crf"
DEVICE = torch.device("cuda")
ARTIFACT_DIR = Path(CFG.artifact_dir)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print(
    "Five-seed retrain protocol:",
    "| seeds:", SEEDS,
    "| graph alpha:", SELECTED_ALPHA,
    "| base auxiliary weight:", SELECTED_BASE_AUX,
    "| patience:", PATIENCE,
    "| train runs:", len(SEEDS),
    "| initialization: pretrained LayoutLMv3 + random heads/graph",
)


Five-seed retrain protocol: | seeds: [13, 42, 2026, 7, 123] | graph alpha: 0.5 | base auxiliary weight: 0.35 | patience: 6 | train runs: 5 | initialization: pretrained LayoutLMv3 + random heads/graph


In [4]:
def set_seed(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def find_cord_root() -> Path:
    candidates = [Path("CORD1000/CORD/CORD")]
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        candidates.extend(path.parent.parent for path in kaggle_input.rglob("train/json"))
    for root in candidates:
        if all(
            (root / split / "json").is_dir() and (root / split / "image").is_dir()
            for split in ("train", "dev", "test")
        ):
            return root.resolve()
    raise FileNotFoundError(
        "Không tìm thấy CORD root có train/dev/test/{image,json}. "
        "Hãy attach dataset CORD-1000 vào Kaggle Notebook."
    )


set_seed(SEEDS[0])
CORD_ROOT = find_cord_root()
print("CORD root:", CORD_ROOT)

CORD root: /kaggle/input/datasets/lonelvino/cord-1000/CORD/CORD


## 3. Label Schema and Dataset Audit

This section maps CORD annotations to 18 receipt entity classes and the background class `O`. It validates image–annotation pairing for every split and summarizes document counts, word counts, and class frequencies.

Entity Macro F1 over the 18 non-background classes is the primary metric. Entity Micro F1, weighted F1, precision, recall, per-class support, and graph-specific diagnostics are retained for complementary analysis.


In [5]:
CORD_CATEGORY_TO_LABEL = {
    "menu.nm": "S-MENU_NM", "menu.sub_nm": "S-MENU_NM",
    "menu.cnt": "S-MENU_CNT", "menu.sub_cnt": "S-MENU_CNT",
    "menu.num": "S-MENU_NUM", "menu.unitprice": "S-MENU_UNITPRICE",
    "menu.price": "S-MENU_PRICE", "menu.sub_price": "S-MENU_PRICE",
    "menu.discountprice": "S-MENU_DISCOUNT_PRICE",
    "sub_total.subtotal_price": "S-SUBTOTAL",
    "sub_total.discount_price": "S-DISCOUNT",
    "sub_total.tax_price": "S-TAX", "sub_total.service_price": "S-SERVICE",
    "total.total_price": "S-TOTAL", "total.cashprice": "S-CASH",
    "total.changeprice": "S-CHANGE", "total.creditcardprice": "S-CARD_PAYMENT",
    "total.emoneyprice": "S-EMONEY_PAYMENT", "total.menuqty_cnt": "S-MENUQTY_CNT",
    "total.menutype_cnt": "S-MENUTYPE_CNT",
    "sub_total.etc": "S-OTHER", "total.total_etc": "S-OTHER",
    "menu.etc": "S-OTHER", "menu.sub_etc": "S-OTHER",
    "menu.itemsubtotal": "O", "menu.vatyn": "O",
    "sub_total.othersvc_price": "O", "void_menu.nm": "O",
    "void_menu.price": "O", "menu.sub_unitprice": "O",
}

LABELS = [
    "O", "S-MENU_NM", "S-MENU_CNT", "S-MENU_NUM", "S-MENU_UNITPRICE",
    "S-MENU_PRICE", "S-MENU_DISCOUNT_PRICE", "S-SUBTOTAL", "S-DISCOUNT",
    "S-TAX", "S-SERVICE", "S-TOTAL", "S-CASH", "S-CHANGE",
    "S-CARD_PAYMENT", "S-EMONEY_PAYMENT", "S-MENUQTY_CNT",
    "S-MENUTYPE_CNT", "S-OTHER",
]
ID2LABEL = dict(enumerate(LABELS))
LABEL2ID = {label: idx for idx, label in ID2LABEL.items()}
ENTITY_IDS = list(range(1, len(LABELS)))
NUM_LABELS = len(LABELS)


def dataset_audit(root: Path) -> pd.DataFrame:
    rows = []
    for split in ("train", "dev", "test"):
        json_files = sorted((root / split / "json").glob("*.json"))
        image_files = sorted((root / split / "image").glob("*.png"))
        json_stems, image_stems = {p.stem for p in json_files}, {p.stem for p in image_files}
        counts = Counter()
        for path in json_files:
            data = json.loads(path.read_text(encoding="utf-8"))
            for line in data.get("valid_line", []):
                mapped = CORD_CATEGORY_TO_LABEL.get(line.get("category", ""), "O")
                counts[mapped] += len(line.get("words", []))
        rows.append({
            "split": split, "json": len(json_files), "images": len(image_files),
            "missing_images": len(json_stems - image_stems),
            "missing_json": len(image_stems - json_stems),
            "words": sum(counts.values()), **counts,
        })
    return pd.DataFrame(rows).fillna(0)


audit_df = dataset_audit(CORD_ROOT)
display(audit_df)
assert (audit_df[["missing_images", "missing_json"]].to_numpy() == 0).all(), "Dataset thiếu cặp image/json."


,split,json,images,missing_images,missing_json,words,S-MENU_CNT,S-MENU_NM,S-MENU_PRICE,S-SUBTOTAL,...,S-CHANGE,S-MENUTYPE_CNT,S-MENUQTY_CNT,S-DISCOUNT,S-MENU_UNITPRICE,S-CARD_PAYMENT,S-MENU_NUM,S-MENU_DISCOUNT_PRICE,S-EMONEY_PAYMENT,O
0,train,800,800,0,0,19371,2126,5995,2236,1187,...,1044,105,513,164,629,326,94,355,115,31
1,dev,100,100,0,0,2186,246,686,241,151,...,135,8,50,11,52,34,4,18,10,3
2,test,100,100,0,0,2356,246,740,268,145,...,120,17,67,16,69,51,11,30,4,6


## 4. Symbolic Graph Construction and Word-Level Inputs

This section builds one graph for each receipt and prepares the corresponding LayoutLMv3 inputs.

Each graph node represents a CORD word. The graph uses seven relation types:

- `SAME_LINE` connects words that share the same `valid_line` identifier.
- `NEXT_LINE_COLUMN` connects aligned words across consecutive lines using horizontal overlap and center-distance constraints.
- `LEFT`, `RIGHT`, `ABOVE`, and `BELOW` encode directional spatial relationships.
- `KNN` supplies local fallback edges when symbolic and directional relations do not fill the neighbor budget.

Every edge also carries continuous geometric attributes derived from relative position, distance, size, overlap, and orientation. Graph construction uses document structure and geometry only; entity labels are never used to create edges.

CORD text and normalized bounding boxes are encoded with `apply_ocr=False`. LayoutLMv3 subtokens are mapped back to words, truncated words are removed consistently from labels and graphs, and variable-size graphs are batched with PyTorch Geometric.


In [6]:
def quad_to_bbox(quad, width, height):
    xs = [quad[f"x{i}"] for i in range(1, 5)]
    ys = [quad[f"y{i}"] for i in range(1, 5)]
    x0, y0, x1, y1 = int(min(xs)), int(min(ys)), int(max(xs)), int(max(ys))
    x0, y0 = max(0, min(x0, width - 1)), max(0, min(y0, height - 1))
    x1, y1 = max(x0 + 1, min(x1, width)), max(y0 + 1, min(y1, height))
    return x0, y0, x1, y1


def build_records(root: Path, split: str):
    records = []
    for json_path in tqdm(sorted((root / split / "json").glob("*.json")), desc=f"Parse {split}"):
        data = json.loads(json_path.read_text(encoding="utf-8"))
        width = int(data["meta"]["image_size"]["width"])
        height = int(data["meta"]["image_size"]["height"])
        image_path = root / split / "image" / f"{json_path.stem}.png"
        words = []
        for line_index, line in enumerate(data.get("valid_line", [])):
            label = CORD_CATEGORY_TO_LABEL.get(line.get("category", ""), "O")
            for word in line.get("words", []):
                text = word.get("text", "").strip()
                if text:
                    raw_row_id = word.get("row_id")
                    line_id = (
                        f"row_{raw_row_id}"
                        if raw_row_id is not None
                        else f"valid_line_{line_index}"
                    )
                    words.append({
                        "text": text,
                        "box": quad_to_bbox(word["quad"], width, height),
                        "label": label,
                        "line_id": line_id,
                    })
        words.sort(key=lambda item: (item["box"][1], item["box"][0]))
        if words and image_path.exists():
            records.append({
                "id": json_path.stem,
                "image_path": image_path,
                "size": (width, height),
                "words": words,
            })
    return records


train_records = build_records(CORD_ROOT, "train")
dev_records = build_records(CORD_ROOT, "dev")
test_records = build_records(CORD_ROOT, "test")
print("Documents:", len(train_records), len(dev_records), len(test_records))

Parse train:   0%|          | 0/800 [00:00<?, ?it/s]

Parse dev:   0%|          | 0/100 [00:00<?, ?it/s]

Parse test:   0%|          | 0/100 [00:00<?, ?it/s]

Documents: 800 100 100


In [7]:
def normalize_box(box, width, height):
    x0, y0, x1, y1 = box
    values = [1000 * x0 / width, 1000 * y0 / height, 1000 * x1 / width, 1000 * y1 / height]
    return [max(0, min(1000, int(v))) for v in values]


def vertical_overlap_ratio(a, b):
    _, ay0, _, ay1 = a
    _, by0, _, by1 = b
    overlap = max(0.0, min(ay1, by1) - max(ay0, by0))
    return overlap / max(min(ay1 - ay0, by1 - by0), 1.0)


def horizontal_overlap_ratio(a, b):
    ax0, _, ax1, _ = a
    bx0, _, bx1, _ = b
    overlap = max(0.0, min(ax1, bx1) - max(ax0, bx0))
    return overlap / max(min(ax1 - ax0, bx1 - bx0), 1.0)


def same_line_geometry(a, b, threshold=0.5):
    return vertical_overlap_ratio(a, b) >= threshold


def horizontally_aligned(a, b, image_width, threshold=0.10):
    ax0, _, ax1, _ = a
    bx0, _, bx1, _ = b
    center_distance = abs((ax0 + ax1 - bx0 - bx1) / 2) / image_width
    return (
        horizontal_overlap_ratio(a, b) >= CFG.column_overlap_threshold
        or center_distance <= threshold
    )


RELATION_TO_ID = {
    "KNN": 0,
    "LEFT": 1,
    "RIGHT": 2,
    "ABOVE": 3,
    "BELOW": 4,
    "SAME_LINE": 5,
    "NEXT_LINE_COLUMN": 6,
}
INVERSE_RELATION = {
    "KNN": "KNN",
    "LEFT": "RIGHT",
    "RIGHT": "LEFT",
    "ABOVE": "BELOW",
    "BELOW": "ABOVE",
    "SAME_LINE": "SAME_LINE",
    "NEXT_LINE_COLUMN": "NEXT_LINE_COLUMN",
}
assert len(RELATION_TO_ID) == CFG.relation_types
assert CFG.edge_dim == 11 + CFG.relation_types


def build_spatial_graph(words, image_size, k=2):
    width, height = image_size
    boxes = np.asarray([item["box"] for item in words], dtype=np.float32)
    n = len(words)
    centers = np.column_stack((
        (boxes[:, 0] + boxes[:, 2]) / (2 * width),
        (boxes[:, 1] + boxes[:, 3]) / (2 * height),
    ))
    widths = (boxes[:, 2] - boxes[:, 0]) / width
    heights = (boxes[:, 3] - boxes[:, 1]) / height
    edge_relations = defaultdict(set)

    def connect(i, j, relation):
        if i == j:
            return
        edge_relations[(i, j)].add(RELATION_TO_ID[relation])
        edge_relations[(j, i)].add(RELATION_TO_ID[INVERSE_RELATION[relation]])

    line_to_nodes = defaultdict(list)
    for index, word in enumerate(words):
        line_to_nodes[word["line_id"]].append(index)
    ordered_lines = sorted(
        line_to_nodes.values(),
        key=lambda nodes: float(np.mean(centers[nodes, 1])),
    )

    for nodes in ordered_lines:
        ordered_nodes = sorted(nodes, key=lambda index: centers[index, 0])
        for left_node, right_node in zip(ordered_nodes[:-1], ordered_nodes[1:]):
            connect(left_node, right_node, "SAME_LINE")
            connect(left_node, right_node, "RIGHT")

    for upper_nodes, lower_nodes in zip(ordered_lines[:-1], ordered_lines[1:]):
        for i in upper_nodes:
            candidates = [
                j for j in lower_nodes
                if horizontally_aligned(
                    boxes[i], boxes[j], width, CFG.column_center_threshold
                )
            ]
            if candidates:
                j = min(candidates, key=lambda node: abs(centers[node, 0] - centers[i, 0]))
                connect(i, j, "NEXT_LINE_COLUMN")
                connect(i, j, "BELOW")

    if n > 1:
        for i in range(n):
            left = [
                j for j in range(n)
                if centers[j, 0] < centers[i, 0]
                and words[j]["line_id"] == words[i]["line_id"]
            ]
            right = [
                j for j in range(n)
                if centers[j, 0] > centers[i, 0]
                and words[j]["line_id"] == words[i]["line_id"]
            ]
            if left:
                connect(i, min(left, key=lambda j: centers[i, 0] - centers[j, 0]), "LEFT")
            if right:
                connect(i, min(right, key=lambda j: centers[j, 0] - centers[i, 0]), "RIGHT")

        for i in range(n):
            current = {dst for src, dst in edge_relations if src == i}
            free_slots = max(CFG.max_graph_neighbors - len(current), 0)
            if free_slots == 0:
                continue
            distances = np.linalg.norm(centers - centers[i], axis=1)
            candidates = [
                int(j) for j in np.argsort(distances)
                if j != i and int(j) not in current
            ]
            for j in candidates[:min(k, free_slots)]:
                connect(i, j, "KNN")

    symbolic_ids = {
        RELATION_TO_ID["SAME_LINE"],
        RELATION_TO_ID["NEXT_LINE_COLUMN"],
    }
    directional_ids = {
        RELATION_TO_ID["LEFT"], RELATION_TO_ID["RIGHT"],
        RELATION_TO_ID["ABOVE"], RELATION_TO_ID["BELOW"],
    }
    bounded_relations = {}
    for i in range(n):
        destinations = [j for src, j in edge_relations if src == i]

        def priority(j):
            relations = edge_relations[(i, j)]
            if relations & symbolic_ids:
                relation_priority = 0
            elif relations & directional_ids:
                relation_priority = 1
            else:
                relation_priority = 2
            return relation_priority, float(np.linalg.norm(centers[j] - centers[i]))

        destinations.sort(key=priority)
        for j in destinations[:CFG.max_graph_neighbors]:
            bounded_relations[(i, j)] = edge_relations[(i, j)]

    src, dst, features = [], [], []
    for (i, j), relation_ids in sorted(bounded_relations.items()):
        dx = float(centers[j, 0] - centers[i, 0])
        dy = float(centers[j, 1] - centers[i, 1])
        distance = float(np.hypot(dx, dy))
        angle = math.atan2(dy, dx)
        log_width_ratio = float(np.clip(
            np.log(max(widths[j], 1e-6) / max(widths[i], 1e-6)), -4, 4
        ))
        log_height_ratio = float(np.clip(
            np.log(max(heights[j], 1e-6) / max(heights[i], 1e-6)), -4, 4
        ))
        log_area_ratio = float(np.clip(
            np.log(max(widths[j] * heights[j], 1e-6) / max(widths[i] * heights[i], 1e-6)),
            -4, 4,
        ))
        relation_one_hot = [
            float(rel_id in relation_ids) for rel_id in range(CFG.relation_types)
        ]
        src.append(i)
        dst.append(j)
        features.append([
            dx, dy, distance,
            log_width_ratio, log_height_ratio, log_area_ratio,
            vertical_overlap_ratio(boxes[i], boxes[j]),
            horizontal_overlap_ratio(boxes[i], boxes[j]),
            float(words[i]["line_id"] == words[j]["line_id"]),
            math.sin(angle), math.cos(angle),
            *relation_one_hot,
        ])

    has_edges = bool(bounded_relations)
    edge_index = (
        torch.tensor([src, dst], dtype=torch.long)
        if has_edges else torch.empty((2, 0), dtype=torch.long)
    )
    edge_attr = (
        torch.tensor(features, dtype=torch.float32)
        if has_edges else torch.empty((0, CFG.edge_dim), dtype=torch.float32)
    )
    spatial = torch.tensor(np.column_stack((centers, widths, heights)), dtype=torch.float32)
    assert edge_attr.shape[1] == CFG.edge_dim
    return Data(edge_index=edge_index, edge_attr=edge_attr, spatial_pos=spatial, num_nodes=n)

In [8]:
processor = LayoutLMv3Processor.from_pretrained(
    CFG.model_id, revision=CFG.model_revision, apply_ocr=False
)


class CORDWordDataset(Dataset):
    def __init__(self, records):
        self.records = records

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]
        width, height = record["size"]
        image = Image.open(record["image_path"]).convert("RGB")
        texts = [item["text"] for item in record["words"]]
        boxes = [normalize_box(item["box"], width, height) for item in record["words"]]
        encoding = processor(
            image, texts, boxes=boxes, truncation=True, max_length=CFG.max_length,
            padding="max_length", return_tensors="pt",
        )
        word_ids = encoding.word_ids(batch_index=0)
        active_ids = sorted({wid for wid in word_ids if wid is not None})
        ranges = []
        for wid in active_ids:
            positions = [pos for pos, current in enumerate(word_ids) if current == wid]
            ranges.append((positions[0], positions[-1] + 1))
        active_words = [record["words"][wid] for wid in active_ids]
        labels = torch.tensor([LABEL2ID[item["label"]] for item in active_words], dtype=torch.long)
        graph = build_spatial_graph(active_words, record["size"], CFG.knn_k)
        return {
            "id": record["id"],
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "bbox": encoding["bbox"].squeeze(0),
            "pixel_values": encoding["pixel_values"].squeeze(0),
            "word_ranges": ranges, "word_labels": labels, "graph": graph,
        }


def collate_fn(samples):
    tensor_keys = ("input_ids", "attention_mask", "bbox", "pixel_values")
    batch = {key: torch.stack([sample[key] for sample in samples]) for key in tensor_keys}
    max_words = max(len(sample["word_labels"]) for sample in samples)
    labels = torch.zeros((len(samples), max_words), dtype=torch.long)
    mask = torch.zeros((len(samples), max_words), dtype=torch.bool)
    for i, sample in enumerate(samples):
        size = len(sample["word_labels"])
        labels[i, :size] = sample["word_labels"]
        mask[i, :size] = True
    batch.update({
        "ids": [sample["id"] for sample in samples],
        "word_ranges": [sample["word_ranges"] for sample in samples],
        "word_labels": labels, "word_mask": mask,
        "graphs": Batch.from_data_list([sample["graph"] for sample in samples]),
    })
    return batch


train_ds, dev_ds, test_ds = map(CORDWordDataset, (train_records, dev_records, test_records))


def build_document_sample_weights(records):
    document_labels = [
        {LABEL2ID[word["label"]] for word in record["words"] if word["label"] != "O"}
        for record in records
    ]
    document_frequency = Counter(label for labels in document_labels for label in labels)
    weights = []
    for labels in document_labels:
        rarity = [
            (len(records) / max(document_frequency[label], 1)) ** CFG.rare_sampler_power
            for label in labels
        ]
        weights.append(min(max(rarity, default=1.0), CFG.rare_sampler_max_weight))
    return torch.tensor(weights, dtype=torch.double), document_frequency


train_sample_weights, train_document_frequency = build_document_sample_weights(train_records)
print("Rare-document sampler:", CFG.use_rare_document_sampler,
      "| weight range:", f"{train_sample_weights.min():.2f}–{train_sample_weights.max():.2f}")


def make_loaders(seed):
    loader_generator = torch.Generator().manual_seed(seed)
    common = dict(batch_size=CFG.batch_size, num_workers=CFG.num_workers,
                  pin_memory=True, collate_fn=collate_fn,
                  persistent_workers=CFG.num_workers > 0)
    if CFG.use_rare_document_sampler:
        sampler_generator = torch.Generator().manual_seed(seed)
        sampler = WeightedRandomSampler(
            train_sample_weights, num_samples=len(train_ds), replacement=True,
            generator=sampler_generator,
        )
        train_loader = DataLoader(train_ds, sampler=sampler, generator=loader_generator, **common)
    else:
        train_loader = DataLoader(train_ds, shuffle=True, generator=loader_generator, **common)
    return (
        train_loader,
        DataLoader(dev_ds, shuffle=False, **common),
        DataLoader(test_ds, shuffle=False, **common),
    )


preprocessor_config.json:   0%|          | 0.00/275 [00:00<?, ?B/s]

The image processor of type `LayoutLMv3ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json:   0%|          | 0.00/856 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Rare-document sampler: True | weight range: 1.05–2.50


## 5. Two-Layer Relation-GATv2, Split Heads, and Hidden Fusion

This section defines the hybrid architecture.

LayoutLMv3 first produces one contextual representation per word. A normalized graph input combines these representations with a small projected spatial signal. Two Relation-GATv2 blocks then perform edge-conditioned message passing. Each block contains pre-normalization, a learned edge encoder, multi-head GATv2 attention, residual connections, and a feed-forward residual sublayer.

The graph branch models a correction rather than replacing the language-layout representation. The change introduced by message passing is projected and fused with the original LayoutLMv3 word state through a fixed-scale residual with `alpha = 0.50`. The final fusion projection is zero-initialized so training begins from the non-graph representation.

Separate linear classifiers are used for the base and fused branches. This prevents the auxiliary base objective from directly shifting the decision boundary of the graph-enhanced classifier. A shared word-level CRF provides structured decoding for both paths. Attention entropy, attention variance, residual magnitude, and residual-to-base norm ratio are recorded as graph diagnostics.


In [9]:
class WordModelBase(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = LayoutLMv3Model.from_pretrained(CFG.model_id, revision=CFG.model_revision)
        self.hidden_size = self.backbone.config.hidden_size
        self.crf = CRF(NUM_LABELS, batch_first=True)

    @staticmethod
    def aggregate_words(token_embeddings, ranges_per_doc):
        docs = []
        for batch_index, ranges in enumerate(ranges_per_doc):
            docs.append(torch.stack([
                token_embeddings[batch_index, start:end].mean(dim=0)
                for start, end in ranges
            ]))
        return docs

    @staticmethod
    def pad_documents(doc_embeddings, max_words):
        hidden = doc_embeddings[0].shape[-1]
        padded = doc_embeddings[0].new_zeros((len(doc_embeddings), max_words, hidden))
        for i, embeddings in enumerate(doc_embeddings):
            padded[i, :len(embeddings)] = embeddings
        return padded

    def encode(self, batch):
        outputs = self.backbone(
            input_ids=batch["input_ids"].to(DEVICE),
            attention_mask=batch["attention_mask"].to(DEVICE),
            bbox=batch["bbox"].to(DEVICE),
            pixel_values=batch["pixel_values"].to(DEVICE),
        )
        text_length = batch["input_ids"].shape[1]
        return self.aggregate_words(outputs.last_hidden_state[:, :text_length], batch["word_ranges"])

    def branch_loss(self, emissions, labels, mask, class_weights):
        emissions_fp32 = emissions.float()
        crf_loss = -self.crf(
            emissions_fp32, labels, mask=mask, reduction="token_mean"
        )
        logits, targets = emissions_fp32[mask], labels[mask]
        ce = F.cross_entropy(logits, targets, weight=class_weights, reduction="none")
        pt = torch.softmax(logits, dim=-1).gather(1, targets[:, None]).squeeze(1)
        focal_loss = (((1 - pt) ** CFG.focal_gamma) * ce).mean()
        return CFG.crf_weight * crf_loss + (1 - CFG.crf_weight) * focal_loss

    def loss(self, emissions, labels, mask, class_weights):
        return self.branch_loss(emissions, labels, mask, class_weights)

    def decode(self, emissions, mask):
        return self.crf.decode(emissions.float(), mask=mask)


class RelationEdgeGATBlock(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        assert hidden_size % CFG.gat_heads == 0
        self.pre_norm = nn.LayerNorm(hidden_size)
        self.edge_encoder = nn.Sequential(
            nn.LayerNorm(CFG.edge_dim),
            nn.Linear(CFG.edge_dim, CFG.gat_edge_hidden),
            nn.GELU(),
            nn.Linear(CFG.gat_edge_hidden, CFG.gat_edge_hidden),
        )
        self.conv = GATv2Conv(
            hidden_size, hidden_size // CFG.gat_heads, heads=CFG.gat_heads,
            concat=True, dropout=CFG.gat_attention_dropout,
            edge_dim=CFG.gat_edge_hidden, add_self_loops=True,
            share_weights=True,
        )
        ffn_hidden = CFG.gat_ffn_multiplier * hidden_size
        self.ffn_norm = nn.LayerNorm(hidden_size)
        self.ffn = nn.Sequential(
            nn.Linear(hidden_size, ffn_hidden), nn.GELU(),
            nn.Dropout(CFG.graph_dropout), nn.Linear(ffn_hidden, hidden_size),
        )
        self.dropout = nn.Dropout(CFG.graph_dropout)
        self.reset_attention_diagnostics()

    def reset_attention_diagnostics(self):
        self._attention_entropy_sum = 0.0
        self._attention_variance_sum = 0.0
        self._attention_batches = 0

    def attention_diagnostics(self):
        if self._attention_batches == 0:
            return {
                "attention_entropy": float("nan"),
                "attention_variance": float("nan"),
            }
        return {
            "attention_entropy": self._attention_entropy_sum / self._attention_batches,
            "attention_variance": self._attention_variance_sum / self._attention_batches,
        }

    def drop_edges(self, edge_index, edge_attr):
        if not self.training or CFG.edge_dropout <= 0 or edge_index.shape[1] == 0:
            return edge_index, edge_attr
        keep = torch.rand(edge_index.shape[1], device=edge_index.device) >= CFG.edge_dropout
        if not bool(keep.any()):
            keep[torch.randint(edge_index.shape[1], (1,), device=edge_index.device)] = True
        return edge_index[:, keep], edge_attr[keep]

    def record_attention(self, attention_edge_index, alpha, num_nodes):
        alpha_fp32 = alpha.detach().float().clamp_min(1e-8)
        destinations = attention_edge_index[1]
        entropy_terms = -alpha_fp32 * alpha_fp32.log()
        node_entropy = alpha_fp32.new_zeros((num_nodes, alpha_fp32.shape[1]))
        node_entropy.index_add_(0, destinations, entropy_terms)
        degree = torch.bincount(destinations, minlength=num_nodes).float()
        valid = degree > 1
        if bool(valid.any()):
            normalizer = degree[valid].log().unsqueeze(-1).clamp_min(1e-8)
            normalized_entropy = node_entropy[valid] / normalizer
            entropy_value = normalized_entropy.mean().item()
        else:
            entropy_value = 0.0
        self._attention_entropy_sum += entropy_value
        self._attention_variance_sum += alpha_fp32.var(unbiased=False).item()
        self._attention_batches += 1

    def forward(self, x, edge_index, edge_attr):
        edge_index, edge_attr = self.drop_edges(edge_index, edge_attr)
        normalized_x = self.pre_norm(x)
        encoded_edge_attr = self.edge_encoder(edge_attr)
        if self.training:
            message = self.conv(normalized_x, edge_index, encoded_edge_attr)
        else:
            message, (attention_edge_index, alpha) = self.conv(
                normalized_x, edge_index, encoded_edge_attr,
                return_attention_weights=True,
            )
            self.record_attention(attention_edge_index, alpha, x.shape[0])
        x = x + CFG.gat_residual_scale * self.dropout(message)
        ffn_message = self.ffn(self.ffn_norm(x))
        return x + CFG.gat_residual_scale * self.dropout(ffn_message)


class LayoutLMv3SymbolicRelationGATFusionCRF(WordModelBase):
    def __init__(self, graph_alpha, base_aux_weight):
        super().__init__()
        hidden = self.hidden_size
        self.register_buffer("_graph_alpha", torch.tensor(float(graph_alpha)))
        self.base_aux_weight = float(base_aux_weight)
        self.spatial_proj = nn.Sequential(
            nn.Linear(4, hidden // 2), nn.GELU(), nn.Linear(hidden // 2, hidden)
        )
        self.graph_input_norm = nn.LayerNorm(hidden)
        self.gat = nn.ModuleList([
            RelationEdgeGATBlock(hidden) for _ in range(CFG.gat_layers)
        ])
        self.graph_proj = nn.Sequential(
            nn.Linear(hidden, hidden), nn.GELU(),
            nn.Dropout(CFG.graph_dropout), nn.Linear(hidden, hidden),
        )
        self.fusion_proj = nn.Sequential(
            nn.Linear(2 * hidden, hidden), nn.GELU(),
            nn.Dropout(CFG.fusion_dropout), nn.Linear(hidden, hidden),
        )
        self.fusion_norm = nn.LayerNorm(hidden)
        self.dropout = nn.Dropout(CFG.lm_dropout)
        self.base_classifier = nn.Linear(hidden, NUM_LABELS)
        self.classifier = nn.Linear(hidden, NUM_LABELS)
        nn.init.zeros_(self.fusion_proj[-1].weight)
        nn.init.zeros_(self.fusion_proj[-1].bias)
        self._current_base_emissions = None
        self.graph_mode = "original"
        self.reset_graph_diagnostics()

    @property
    def graph_alpha(self):
        return float(self._graph_alpha.item())

    def set_backbone_trainable(self, trainable):
        for parameter in self.backbone.parameters():
            parameter.requires_grad = trainable
        if not trainable:
            self.backbone.eval()

    def set_graph_mode(self, mode):
        valid = {"original", "shuffled", "self_loop_only", "graph_off"}
        if mode not in valid:
            raise ValueError(f"Unknown graph mode: {mode}")
        self.graph_mode = mode

    @staticmethod
    def shuffled_edges(graph):
        edge_index = graph.edge_index
        if edge_index.shape[1] == 0:
            return edge_index, graph.edge_attr
        node_map = torch.arange(graph.num_nodes, device=edge_index.device)
        boundaries = graph.ptr.detach().cpu().tolist()
        for doc_index, (start, end) in enumerate(zip(boundaries[:-1], boundaries[1:])):
            size = end - start
            generator = torch.Generator().manual_seed(
                CFG.graph_shuffle_seed + 1009 * size + doc_index
            )
            permutation = torch.randperm(size, generator=generator).to(edge_index.device) + start
            node_map[start:end] = permutation
        return node_map[edge_index], graph.edge_attr

    def ablation_edges(self, graph):
        if self.graph_mode == "original":
            return graph.edge_index, graph.edge_attr
        if self.graph_mode == "shuffled":
            return self.shuffled_edges(graph)
        if self.graph_mode == "self_loop_only":
            return (
                torch.empty((2, 0), dtype=torch.long, device=graph.edge_index.device),
                torch.empty((0, CFG.edge_dim), dtype=graph.edge_attr.dtype, device=graph.edge_attr.device),
            )
        raise RuntimeError("graph_off does not request graph edges")

    def reset_graph_diagnostics(self):
        self._base_norm_sum = 0.0
        self._residual_norm_sum = 0.0
        self._diagnostic_token_count = 0
        for layer in self.gat:
            layer.reset_attention_diagnostics()

    def graph_diagnostics(self):
        diagnostics = {"graph_alpha": self.graph_alpha}
        if self._diagnostic_token_count == 0:
            diagnostics.update({
                "graph_residual_base_ratio": float("nan"),
                "graph_residual_norm": float("nan"),
            })
        else:
            diagnostics.update({
                "graph_residual_base_ratio": (
                    self._residual_norm_sum / max(self._base_norm_sum, 1e-12)
                ),
                "graph_residual_norm": (
                    self._residual_norm_sum / self._diagnostic_token_count
                ),
            })
        for layer_index, layer in enumerate(self.gat, start=1):
            for name, value in layer.attention_diagnostics().items():
                diagnostics[f"layer{layer_index}_{name}"] = value
        return diagnostics

    def loss(self, emissions, labels, mask, class_weights):
        hybrid_loss = self.branch_loss(emissions, labels, mask, class_weights)
        if self._current_base_emissions is None or self.base_aux_weight <= 0:
            return hybrid_loss
        base_loss = self.branch_loss(
            self._current_base_emissions, labels, mask, class_weights
        )
        return (
            hybrid_loss + self.base_aux_weight * base_loss
        ) / (1 + self.base_aux_weight)

    def forward(self, batch):
        lm_docs = self.encode(batch)
        lm_words = torch.cat(lm_docs, dim=0)
        base_flat = self.base_classifier(self.dropout(lm_words))

        base_docs, offset = [], 0
        for lm_doc in lm_docs:
            base_docs.append(base_flat[offset:offset + len(lm_doc)])
            offset += len(lm_doc)
        base_padded = self.pad_documents(base_docs, batch["word_mask"].shape[1])
        self._current_base_emissions = base_padded

        branch_dropped = (
            self.training
            and self.graph_mode == "original"
            and bool(torch.rand((), device=lm_words.device) < CFG.graph_branch_dropout)
        )
        if self.graph_mode == "graph_off" or branch_dropped:
            return base_padded

        graph = batch["graphs"].to(DEVICE)
        edge_index, edge_attr = self.ablation_edges(graph)
        graph_seed = self.graph_input_norm(
            lm_words + CFG.spatial_input_scale * self.spatial_proj(graph.spatial_pos)
        )
        graph_words = graph_seed
        for layer in self.gat:
            graph_words = layer(graph_words, edge_index, edge_attr)

        graph_delta = self.graph_proj(graph_words - graph_seed)
        fusion_residual = self.fusion_proj(torch.cat([lm_words, graph_delta], dim=-1))
        scaled_residual = self.graph_alpha * fusion_residual
        fused_words = self.fusion_norm(lm_words + scaled_residual)
        flat_emissions = self.classifier(self.dropout(fused_words))

        if not self.training:
            detached_base = lm_words.detach().float()
            detached_residual = scaled_residual.detach().float()
            self._base_norm_sum += detached_base.norm(dim=-1).sum().item()
            self._residual_norm_sum += detached_residual.norm(dim=-1).sum().item()
            self._diagnostic_token_count += detached_base.shape[0]

        docs, offset = [], 0
        for lm_doc in lm_docs:
            docs.append(flat_emissions[offset:offset + len(lm_doc)])
            offset += len(lm_doc)
        return self.pad_documents(docs, batch["word_mask"].shape[1])

## 6. Training Objective, Optimization, and Evaluation

The fused branch and auxiliary base branch each use a combination of CRF negative log-likelihood and class-weighted focal cross-entropy. Their losses are combined as:

```text
(fusion_loss + 0.35 × base_loss) / 1.35
```

The auxiliary term maintains a useful LayoutLMv3-only representation while allowing the fused graph path to remain the primary optimization target.

Class weights are computed exclusively from the training split using capped effective-number weighting. The optimizer assigns separate learning rates to the backbone, task heads, and graph modules. Training uses mixed precision, gradient accumulation, gradient clipping, linear warmup and decay, one epoch of backbone freezing, and early stopping with patience `6`.

Checkpoint selection uses development Entity Macro F1 in the `original` graph mode only. Epoch-level graph-off measurements and attention statistics are diagnostic outputs and do not influence early stopping or checkpoint selection.


In [10]:
train_label_counts = Counter(
    LABEL2ID[word["label"]] for record in train_records for word in record["words"]
)
counts = torch.tensor([train_label_counts.get(i, 0) for i in range(NUM_LABELS)], dtype=torch.float32)
beta = CFG.class_balance_beta
effective_counts = (1 - torch.pow(torch.tensor(beta), counts.clamp_min(1))) / (1 - beta)
class_weights = effective_counts.reciprocal()
entity_mean = class_weights[1:].mean()
class_weights = class_weights / entity_mean
class_weights[0] = 1.0
class_weights = class_weights.clamp(max=CFG.class_weight_cap).to(DEVICE)
display(pd.DataFrame({"label": LABELS, "train_words": counts.int(), "weight": class_weights.cpu()}))


def calculate_metrics(golds, preds):
    precision, recall, macro_f1, _ = precision_recall_fscore_support(
        golds, preds, labels=ENTITY_IDS, average="macro", zero_division=0
    )
    return {
        "entity_macro_precision": float(precision),
        "entity_macro_recall": float(recall),
        "entity_macro_f1": float(macro_f1),
        "entity_micro_f1": float(f1_score(golds, preds, labels=ENTITY_IDS, average="micro", zero_division=0)),
        "weighted_f1": float(f1_score(golds, preds, labels=list(range(NUM_LABELS)), average="weighted", zero_division=0)),
    }


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    if hasattr(model, "reset_graph_diagnostics"):
        model.reset_graph_diagnostics()
    losses, golds, preds = [], [], []
    for batch in tqdm(loader, leave=False, desc="evaluate"):
        labels = batch["word_labels"].to(DEVICE)
        mask = batch["word_mask"].to(DEVICE)
        with torch.amp.autocast(device_type="cuda", enabled=CFG.use_amp):
            emissions = model(batch)
            loss = model.loss(emissions, labels, mask, class_weights)
        losses.append(loss.item())
        paths = model.decode(emissions, mask)
        for i, path in enumerate(paths):
            length = int(mask[i].sum())
            preds.extend(path[:length])
            golds.extend(labels[i, :length].tolist())
    metrics = calculate_metrics(golds, preds)
    metrics["loss"] = float(np.mean(losses))
    if hasattr(model, "graph_diagnostics"):
        metrics.update(model.graph_diagnostics())
    report = classification_report(
        golds, preds, labels=list(range(NUM_LABELS)), target_names=LABELS,
        zero_division=0, output_dict=True,
    )
    return metrics, report, golds, preds


def parameter_groups(model):
    groups = {"backbone": [], "base_head": [], "graph": []}
    for name, parameter in model.named_parameters():
        if name.startswith("backbone."):
            groups["backbone"].append(parameter)
        elif name.startswith(("spatial_proj.", "graph_input_norm.", "gat.", "graph_proj.", "fusion_proj.", "fusion_norm.")):
            groups["graph"].append(parameter)
        else:
            groups["base_head"].append(parameter)
    specs = [
        ("backbone", CFG.backbone_lr, CFG.weight_decay),
        ("base_head", CFG.head_lr, CFG.weight_decay),
        ("graph", CFG.graph_lr, CFG.weight_decay),
    ]
    return [
        {"params": groups[name], "lr": learning_rate, "weight_decay": weight_decay}
        for name, learning_rate, weight_decay in specs if groups[name]
    ]


def train_model(
    model, train_loader, dev_loader, run_name, seed, patience
):
    model.to(DEVICE)
    optimizer = AdamW(parameter_groups(model))
    steps_per_epoch = math.ceil(len(train_loader) / CFG.grad_accum_steps)
    total_steps = steps_per_epoch * MAX_EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer, int(total_steps * CFG.warmup_ratio), total_steps
    )
    scaler = torch.amp.GradScaler("cuda", enabled=CFG.use_amp)
    checkpoint_path = ARTIFACT_DIR / f"{run_name}_seed{seed}.pt"
    history, best_f1, best_epoch, stale_epochs = [], -1.0, -1, 0

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        model.set_graph_mode("original")
        backbone_trainable = epoch > CFG.freeze_backbone_epochs
        model.set_backbone_trainable(backbone_trainable)
        optimizer.zero_grad(set_to_none=True)
        epoch_losses = []
        progress = tqdm(train_loader, desc=f"{run_name} seed={seed} epoch={epoch}")
        for step, batch in enumerate(progress, start=1):
            labels = batch["word_labels"].to(DEVICE)
            mask = batch["word_mask"].to(DEVICE)
            with torch.amp.autocast(device_type="cuda", enabled=CFG.use_amp):
                emissions = model(batch)
                loss = model.loss(emissions, labels, mask, class_weights)
                scaled_loss = loss / CFG.grad_accum_steps
            scaler.scale(scaled_loss).backward()
            if step % CFG.grad_accum_steps == 0 or step == len(train_loader):
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scale_before = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                if scaler.get_scale() >= scale_before:
                    scheduler.step()
                optimizer.zero_grad(set_to_none=True)
            epoch_losses.append(loss.item())
            progress.set_postfix(loss=f"{np.mean(epoch_losses[-20:]):.4f}")

        model.set_graph_mode("original")
        dev_metrics, _, _, _ = evaluate(model, dev_loader)
        epoch_graph_off_metrics = {}
        if CFG.track_epoch_graph_off:
            model.set_graph_mode("graph_off")
            graph_off_metrics, _, _, _ = evaluate(model, dev_loader)
            epoch_graph_off_metrics = {
                f"dev_epoch_graph_off_{key}": value
                for key, value in graph_off_metrics.items()
            }
            model.set_graph_mode("original")
        row = {
            "epoch": epoch,
            "backbone_trainable": backbone_trainable,
            "train_loss": float(np.mean(epoch_losses)),
            **{f"dev_{key}": value for key, value in dev_metrics.items()},
            **epoch_graph_off_metrics,
        }
        history.append(row)
        print(row)

        current_f1 = dev_metrics["entity_macro_f1"]
        if current_f1 > best_f1 + CFG.min_delta:
            best_f1, best_epoch, stale_epochs = current_f1, epoch, 0
            torch.save(model.state_dict(), checkpoint_path)
        else:
            stale_epochs += 1
            if stale_epochs >= patience:
                print(f"Early stopping at epoch {epoch}.")
                break

    state = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    model.load_state_dict(state, strict=True)
    model.to(DEVICE).eval()
    pd.DataFrame(history).to_csv(
        ARTIFACT_DIR / f"{run_name}_seed{seed}_history.csv", index=False
    )
    return model, checkpoint_path, history, best_epoch


,label,train_words,weight
0,O,31,1.000000
1,S-MENU_NM,5994,0.270638
2,S-MENU_CNT,2126,0.306496
3,S-MENU_NUM,94,3.000000
4,S-MENU_UNITPRICE,629,0.578035
5,S-MENU_PRICE,2236,0.302234
6,S-MENU_DISCOUNT_PRICE,355,0.903049
7,S-SUBTOTAL,1187,0.388415
8,S-DISCOUNT,164,1.783996
9,S-TAX,1022,0.421619


## 7. Five-Seed Training, Graph Ablations, and Locked Test Evaluation

This section executes five independent training runs, one for each configured seed. For every run, the best checkpoint is selected using development Entity Macro F1 under the original graph topology.

After checkpoint selection, the development split is evaluated under four controlled conditions:

- `original`: the complete symbolic relation graph.
- `shuffled`: node connectivity is permuted within each document while edge attributes are retained.
- `self_loop_only`: inter-word message-passing edges are removed.
- `graph_off`: the graph branch is bypassed and predictions use the separate base classifier.

These ablations distinguish gains caused by meaningful graph topology from gains caused by additional model capacity. Graph evidence is considered supportive when the mean original-minus-graph-off contribution exceeds its sample standard deviation and the original topology outperforms both shuffled and self-loop conditions.

The test split is evaluated only after all five seeds have completed training and all four development ablations. This section also saves per-document predictions, applies the predeclared decision rules, records graph-off behavior across epochs, and computes absolute document-level bootstrap intervals.


In [11]:
BASELINE_DEV_F1 = 0.9256
BASELINE_TEST_MACRO_F1 = 0.9233
COLLAPSE_THRESHOLD = BASELINE_DEV_F1 - 0.015
DEV_GAP_LIMIT = 0.003

PREDECLARED_DECISION_RULES = {
    "rule_A_supportive_hybrid": {
        "graph_contribution": "mean > sample std across five seeds",
        "gap_vs_baseline_dev_reference": (
            "baseline reported dev mean - hybrid five-seed dev mean < 0.003"
        ),
        "test_reference": (
            "five-seed mean test macro F1 > reported baseline mean 0.9233"
        ),
        "conclusion": (
            "Hybrid has supportive five-seed evidence; paired baseline "
            "checkpoints are still required for a strict superiority claim."
        ),
    },
    "rule_B_systematic_base_problem": {
        "collapse": (
            f"at least two seeds have dev graph-off < "
            f"{COLLAPSE_THRESHOLD:.4f}"
        ),
        "graph_contribution": "mean <= sample std",
        "conclusion": (
            "Base-branch collapse is systematic; investigate optimization."
        ),
    },
    "rule_C_hybrid_not_reliable": {
        "gap_vs_baseline_dev_reference": (
            "reported baseline dev mean - hybrid dev mean >= 0.003"
        ),
        "conclusion": (
            "Hybrid is not reliably competitive with the baseline reference."
        ),
    },
}
(ARTIFACT_DIR / "predeclared_decision_rules.json").write_text(
    json.dumps(PREDECLARED_DECISION_RULES, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("PREDECLARED DECISION RULES")
display(pd.Series(PREDECLARED_DECISION_RULES))


@torch.no_grad()
def predict_documents(model, loader):
    model.eval()
    if hasattr(model, "set_graph_mode"):
        model.set_graph_mode("original")
    documents = {}
    for batch in tqdm(loader, leave=False, desc="document predictions"):
        labels = batch["word_labels"].to(DEVICE)
        mask = batch["word_mask"].to(DEVICE)
        with torch.amp.autocast(device_type="cuda", enabled=CFG.use_amp):
            emissions = model(batch)
        paths = model.decode(emissions, mask)
        for index, path in enumerate(paths):
            length = int(mask[index].sum())
            documents[batch["ids"][index]] = {
                "gold": labels[index, :length].tolist(),
                "pred": list(path[:length]),
            }
    return documents


def document_bootstrap_absolute(
    documents_by_seed,
    split,
    repetitions,
    bootstrap_seed,
    reference_macro_f1,
):
    document_ids = sorted(next(iter(documents_by_seed.values())))
    for seed, documents in documents_by_seed.items():
        if sorted(documents) != document_ids:
            raise AssertionError(f"Document mismatch for seed {seed}")
        for document_id in document_ids:
            reference_gold = next(iter(documents_by_seed.values()))[
                document_id
            ]["gold"]
            if documents[document_id]["gold"] != reference_gold:
                raise AssertionError(f"Gold mismatch for {document_id}")

    rng = np.random.default_rng(bootstrap_seed)
    sampled_indices = rng.integers(
        0, len(document_ids), size=(repetitions, len(document_ids))
    )
    values_by_seed = {
        int(seed): np.empty(repetitions, dtype=np.float32)
        for seed in documents_by_seed
    }
    for repetition, indices in enumerate(sampled_indices):
        sampled_ids = [document_ids[index] for index in indices]
        gold = [
            label
            for document_id in sampled_ids
            for label in next(iter(documents_by_seed.values()))[
                document_id
            ]["gold"]
        ]
        for seed, documents in documents_by_seed.items():
            prediction = [
                label
                for document_id in sampled_ids
                for label in documents[document_id]["pred"]
            ]
            values_by_seed[int(seed)][repetition] = f1_score(
                gold, prediction, labels=ENTITY_IDS,
                average="macro", zero_division=0,
            )

    rows = []
    for seed, values in values_by_seed.items():
        rows.append({
            "split": split,
            "comparison": f"hybrid_seed_{seed}_absolute",
            "seeds": 1,
            "macro_f1_mean": float(values.mean()),
            "ci95_low": float(np.quantile(values, 0.025)),
            "ci95_high": float(np.quantile(values, 0.975)),
            "reference_macro_f1": reference_macro_f1,
            "delta_vs_reference_mean": float(
                values.mean() - reference_macro_f1
            ),
            "probability_above_reference": float(
                (values > reference_macro_f1).mean()
            ),
            "bootstrap_documents": len(document_ids),
            "bootstrap_repetitions": repetitions,
        })
    stacked = np.stack(list(values_by_seed.values()), axis=0)
    aggregate = stacked.mean(axis=0)
    rows.append({
        "split": split,
        "comparison": "mean_hybrid_across_five_seeds_absolute",
        "seeds": len(values_by_seed),
        "macro_f1_mean": float(aggregate.mean()),
        "ci95_low": float(np.quantile(aggregate, 0.025)),
        "ci95_high": float(np.quantile(aggregate, 0.975)),
        "reference_macro_f1": reference_macro_f1,
        "delta_vs_reference_mean": float(
            aggregate.mean() - reference_macro_f1
        ),
        "probability_above_reference": float(
            (aggregate > reference_macro_f1).mean()
        ),
        "bootstrap_documents": len(document_ids),
        "bootstrap_repetitions": repetitions,
    })
    return pd.DataFrame(rows)


def alpha_tag(alpha):
    return f"a{int(round(float(alpha) * 100)):03d}"


def aux_tag(weight):
    return f"b{int(round(float(weight) * 100)):03d}"


def run_dev_ablation(model, dev_loader):
    mode_metrics = {}
    for graph_mode in ("original", "shuffled", "self_loop_only", "graph_off"):
        model.set_graph_mode(graph_mode)
        metrics, report, golds, preds = evaluate(model, dev_loader)
        mode_metrics[graph_mode] = {
            "metrics": metrics,
            "report": report,
            "golds": golds,
            "preds": preds,
        }
    model.set_graph_mode("original")
    return mode_metrics


def configuration_summary(rows):
    summary = pd.DataFrame([{
        "model": MODEL_NAME,
        "seeds": len(rows),
        "dev_macro_f1_mean": rows["dev_entity_macro_f1"].mean(),
        "dev_macro_f1_std": rows["dev_entity_macro_f1"].std(ddof=1),
        "dev_graph_off_mean": rows["dev_graph_off_entity_macro_f1"].mean(),
        "dev_graph_off_std": rows["dev_graph_off_entity_macro_f1"].std(ddof=1),
        "best_epoch_mean": rows["best_epoch"].mean(),
        "best_epoch_std": rows["best_epoch"].std(ddof=1),
        "graph_contribution_pp_mean": rows["dev_graph_contribution_pp"].mean(),
        "graph_contribution_pp_std": rows["dev_graph_contribution_pp"].std(ddof=1),
        "original_minus_shuffled_pp_mean": rows[
            "dev_original_minus_shuffled_pp"
        ].mean(),
        "original_minus_shuffled_pp_std": rows[
            "dev_original_minus_shuffled_pp"
        ].std(ddof=1),
        "original_minus_self_loop_pp_mean": rows[
            "dev_original_minus_self_loop_pp"
        ].mean(),
        "original_minus_self_loop_pp_std": rows[
            "dev_original_minus_self_loop_pp"
        ].std(ddof=1),
        "wall_minutes_mean": rows["wall_seconds"].mean() / 60,
    }])
    summary["graph_evidence_pass"] = (
        (
            summary["graph_contribution_pp_mean"]
            > summary["graph_contribution_pp_std"]
        )
        & (summary["original_minus_shuffled_pp_mean"] > 0)
        & (summary["original_minus_self_loop_pp_mean"] > 0)
    )
    summary["gap_vs_baseline_dev_pp"] = 100 * (
        summary["dev_macro_f1_mean"] - BASELINE_DEV_F1
    )
    return summary


def run_configuration(
    stage, configuration, graph_alpha, base_aux_weight, patience, seeds
):
    cache_path = ARTIFACT_DIR / f"{configuration}_seed_cache.csv"
    rows = []
    if cache_path.exists():
        cached = pd.read_csv(cache_path)
        for _, cached_row in cached.iterrows():
            checkpoint = Path(cached_row["checkpoint"])
            if int(cached_row["seed"]) in seeds and checkpoint.exists():
                rows.append(cached_row.to_dict())
    run_name = (
        f"{MODEL_NAME}_{configuration}_{alpha_tag(graph_alpha)}_"
        f"{aux_tag(base_aux_weight)}_p{patience}"
    )
    completed = {int(row["seed"]) for row in rows}
    for seed in seeds:
        if seed in completed:
            print("Reusing cached new seed:", seed)
            continue
        print("\n" + "=" * 110)
        print(stage, "|", configuration, "| NEW seed", seed, "| DEV ONLY")
        print("=" * 110)
        set_seed(seed)
        train_loader, dev_loader, _ = make_loaders(seed)
        torch.cuda.reset_peak_memory_stats()
        model = LayoutLMv3SymbolicRelationGATFusionCRF(
            graph_alpha=graph_alpha,
            base_aux_weight=base_aux_weight,
        )
        total_parameters = sum(
            parameter.numel() for parameter in model.parameters()
        )
        trainable_parameters = sum(
            parameter.numel() for parameter in model.parameters()
            if parameter.requires_grad
        )
        started = time.time()
        model, checkpoint_path, history, best_epoch = train_model(
            model, train_loader, dev_loader, run_name, seed, patience
        )
        diagnostic_started = time.time()
        ablation = run_dev_ablation(model, dev_loader)
        diagnostic_seconds = time.time() - diagnostic_started
        original = ablation["original"]["metrics"]
        shuffled = ablation["shuffled"]["metrics"]
        self_loop = ablation["self_loop_only"]["metrics"]
        graph_off = ablation["graph_off"]["metrics"]
        row = {
            "stage": stage,
            "configuration": configuration,
            "model": MODEL_NAME,
            "seed": seed,
            "graph_alpha": float(graph_alpha),
            "base_aux_weight": float(base_aux_weight),
            "patience": int(patience),
            "checkpoint": str(checkpoint_path),
            "source": "trained_from_scratch",
            "initialization": (
                "pretrained_layoutlmv3_plus_random_graph_no_warmstart"
            ),
            "backbone_lr": CFG.backbone_lr,
            "head_lr": CFG.head_lr,
            "graph_lr": CFG.graph_lr,
            "epochs_ran": len(history),
            "best_epoch": int(best_epoch),
            "total_parameters": total_parameters,
            "trainable_parameters": trainable_parameters,
            "wall_seconds": time.time() - started,
            "diagnostic_seconds": diagnostic_seconds,
            "peak_vram_gb": (
                torch.cuda.max_memory_allocated() / (1024 ** 3)
            ),
            "dev_shuffled_entity_macro_f1": shuffled["entity_macro_f1"],
            "dev_self_loop_entity_macro_f1": self_loop["entity_macro_f1"],
            "dev_graph_off_entity_macro_f1": graph_off["entity_macro_f1"],
            "dev_original_minus_shuffled_pp": 100 * (
                original["entity_macro_f1"] - shuffled["entity_macro_f1"]
            ),
            "dev_original_minus_self_loop_pp": 100 * (
                original["entity_macro_f1"] - self_loop["entity_macro_f1"]
            ),
            "dev_graph_contribution_pp": 100 * (
                original["entity_macro_f1"] - graph_off["entity_macro_f1"]
            ),
            **{f"dev_{key}": value for key, value in original.items()},
        }
        rows.append(row)
        pd.DataFrame(rows).sort_values("seed").to_csv(
            cache_path, index=False
        )
        print(pd.Series(row))
        del model, train_loader, dev_loader
        gc.collect()
        torch.cuda.empty_cache()
    frame = pd.DataFrame(rows).sort_values("seed").reset_index(drop=True)
    assert set(frame["seed"].astype(int)) == set(seeds)
    return frame


experiment_started = time.time()
selected_alpha = float(SELECTED_ALPHA)
selected_base_aux = float(SELECTED_BASE_AUX)

dev_sweep_df = run_configuration(
    stage="five_seed_retrain_from_scratch",
    configuration="latest_frozen_a050_b035_five_seed_retrain",
    graph_alpha=selected_alpha,
    base_aux_weight=selected_base_aux,
    patience=PATIENCE,
    seeds=SEEDS,
)
dev_sweep_df = dev_sweep_df.sort_values("seed").reset_index(drop=True)
assert set(dev_sweep_df["seed"].astype(int)) == set(SEEDS)
assert dev_sweep_df["source"].eq("trained_from_scratch").all()
dev_sweep_df.to_csv(
    ARTIFACT_DIR / "dev_ablation_results.csv", index=False
)
configuration_result = configuration_summary(dev_sweep_df)
configuration_result.to_csv(
    ARTIFACT_DIR / "dev_configuration_summary.csv", index=False
)
display(configuration_result)

# Graph-off trajectory diagnosis for every seed, including seed 13.
epoch_diagnostics = []
for _, row in dev_sweep_df.iterrows():
    seed = int(row["seed"])
    history_path = ARTIFACT_DIR / (
        f"{Path(row['checkpoint']).stem}_history.csv"
    )
    history = pd.read_csv(history_path)
    graph_off_column = "dev_epoch_graph_off_entity_macro_f1"
    if graph_off_column not in history:
        raise AssertionError(f"Missing graph-off history for seed {seed}")
    graph_off_change = history[graph_off_column].diff()
    largest_drop_index = graph_off_change.idxmin()
    epoch_diagnostics.append({
        "seed": seed,
        "best_original_epoch": int(row["best_epoch"]),
        "selected_checkpoint_graph_off_f1": float(
            row["dev_graph_off_entity_macro_f1"]
        ),
        "lowest_graph_off_f1": float(history[graph_off_column].min()),
        "lowest_graph_off_epoch": int(
            history.loc[history[graph_off_column].idxmin(), "epoch"]
        ),
        "largest_graph_off_drop": float(
            graph_off_change.loc[largest_drop_index]
        ),
        "largest_graph_off_drop_epoch": int(
            history.loc[largest_drop_index, "epoch"]
        ),
        "sudden_graph_off_drop_ge_1pp": bool(
            graph_off_change.loc[largest_drop_index] <= -0.01
        ),
    })
epoch_diagnostics_df = pd.DataFrame(epoch_diagnostics).sort_values("seed")
epoch_diagnostics_df.to_csv(
    ARTIFACT_DIR / "epoch_graph_off_diagnostics.csv", index=False
)
display(epoch_diagnostics_df)

# Test unlock: only after all five seeds and all dev ablations completed.
selected_rows = []
reports = {}
hybrid_dev_documents = {}
hybrid_test_documents = {}
for seed in SEEDS:
    selected_seed_row = dev_sweep_df[
        dev_sweep_df["seed"].eq(seed)
    ].iloc[0].to_dict()
    checkpoint = Path(selected_seed_row["checkpoint"])
    set_seed(seed)
    _, dev_loader, test_loader = make_loaders(seed)
    model = LayoutLMv3SymbolicRelationGATFusionCRF(
        selected_alpha, selected_base_aux
    ).to(DEVICE)
    state = torch.load(checkpoint, map_location="cpu", weights_only=True)
    model.load_state_dict(state, strict=True)
    model.to(DEVICE).eval()
    model.set_graph_mode("original")
    dev_metrics, dev_report, _, _ = evaluate(model, dev_loader)
    test_metrics, test_report, _, _ = evaluate(model, test_loader)
    hybrid_dev_documents[seed] = predict_documents(model, dev_loader)
    hybrid_test_documents[seed] = predict_documents(model, test_loader)
    selected_seed_row.update({
        **{f"dev_{key}": value for key, value in dev_metrics.items()},
        **{f"test_{key}": value for key, value in test_metrics.items()},
    })
    selected_rows.append(selected_seed_row)
    reports[f"{MODEL_NAME}_seed{seed}"] = {
        "dev": dev_report,
        "test": test_report,
    }
    del model, dev_loader, test_loader
    gc.collect()
    torch.cuda.empty_cache()

results_df = pd.DataFrame(selected_rows).sort_values("seed").reset_index(drop=True)
results_df.to_csv(ARTIFACT_DIR / "experiment_results.csv", index=False)
(ARTIFACT_DIR / "classification_reports.json").write_text(
    json.dumps(reports, ensure_ascii=False, indent=2), encoding="utf-8"
)

bootstrap_dev_df = document_bootstrap_absolute(
    hybrid_dev_documents,
    split="dev_five_seed",
    repetitions=CFG.bootstrap_repetitions,
    bootstrap_seed=20260727,
    reference_macro_f1=BASELINE_DEV_F1,
)
bootstrap_test_df = document_bootstrap_absolute(
    hybrid_test_documents,
    split="test_five_seed",
    repetitions=CFG.bootstrap_repetitions,
    bootstrap_seed=20260728,
    reference_macro_f1=BASELINE_TEST_MACRO_F1,
)
bootstrap_results_df = pd.concat(
    [bootstrap_dev_df, bootstrap_test_df], ignore_index=True
)
bootstrap_results_df.to_csv(
    ARTIFACT_DIR / "bootstrap_hybrid_absolute_results.csv", index=False
)
display(bootstrap_results_df)

(ARTIFACT_DIR / "document_predictions.json").write_text(
    json.dumps(
        {
            "hybrid_dev": hybrid_dev_documents,
            "hybrid_test": hybrid_test_documents,
        },
        ensure_ascii=False,
        default=lambda value: int(value) if isinstance(value, np.integer) else value,
    ),
    encoding="utf-8",
)
total_experiment_minutes = (time.time() - experiment_started) / 60
print("Total five-seed retrain time (min):", total_experiment_minutes)


PREDECLARED DECISION RULES


rule_A_supportive_hybrid          {'graph_contribution': 'mean > sample std acro...
rule_B_systematic_base_problem    {'collapse': 'at least two seeds have dev grap...
rule_C_hybrid_not_reliable        {'gap_vs_baseline_dev_reference': 'reported ba...
dtype: object


five_seed_retrain_from_scratch | latest_frozen_a050_b035_five_seed_retrain | NEW seed 13 | DEV ONLY


model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

...

evaluate:   0%|          | 0/50 [00:00<?, ?it/s]

document predictions:   0%|          | 0/50 [00:00<?, ?it/s]

document predictions:   0%|          | 0/50 [00:00<?, ?it/s]

,split,comparison,seeds,macro_f1_mean,ci95_low,ci95_high,reference_macro_f1,delta_vs_reference_mean,probability_above_reference,bootstrap_documents,bootstrap_repetitions
0,dev_five_seed,hybrid_seed_13_absolute,1,0.904773,0.830757,0.952239,0.9256,-0.020827,0.2620,100,10000
1,dev_five_seed,hybrid_seed_42_absolute,1,0.908189,0.837794,0.959415,0.9256,-0.017411,0.3253,100,10000
2,dev_five_seed,hybrid_seed_2026_absolute,1,0.905260,0.840216,0.957091,0.9256,-0.020340,0.2636,100,10000
3,dev_five_seed,hybrid_seed_7_absolute,1,0.906276,0.834987,0.959645,0.9256,-0.019324,0.3013,100,10000
4,dev_five_seed,hybrid_seed_123_absolute,1,0.915862,0.849098,0.961846,0.9256,-0.009738,0.4270,100,10000
5,dev_five_seed,mean_hybrid_across_five_seeds_absolute,5,0.908072,0.843762,0.951024,0.9256,-0.017528,0.2912,100,10000
6,test_five_seed,hybrid_seed_13_absolute,1,0.946679,0.891770,0.978203,0.9233,0.023379,0.8435,100,10000
7,test_five_seed,hybrid_seed_42_absolute,1,0.934653,0.892819,0.964289,0.9233,0.011353,0.7518,100,10000
8,test_five_seed,hybrid_seed_2026_absolute,1,0.908150,0.865763,0.943366,0.9233,-0.015150,0.2288,100,10000
9,test_five_seed,hybrid_seed_7_absolute,1,0.937862,0.892558,0.965826,0.9233,0.014562,0.7972,100,10000


Total five-seed retrain time (min): 192.5704911271731


In [12]:
def mean_std_text(values, digits=4):
    values = pd.Series(values, dtype=float).dropna()
    mean = values.mean()
    if len(values) < 2:
        return f"{mean:.{digits}f} (1 seed)"
    return f"{mean:.{digits}f} ± {values.std(ddof=1):.{digits}f}"


group = results_df
final_summary = configuration_summary(group).iloc[0]
final_table = pd.DataFrame([{
    "Model": MODEL_NAME,
    "Seeds": len(group),
    "Train runs from scratch": len(SEEDS),
    "Dev Entity Macro F1": mean_std_text(group["dev_entity_macro_f1"]),
    "Dev graph-off Macro F1": mean_std_text(
        group["dev_graph_off_entity_macro_f1"]
    ),
    "Test Entity Macro F1": mean_std_text(group["test_entity_macro_f1"]),
    "Test Entity Micro F1": mean_std_text(group["test_entity_micro_f1"]),
    "Test Weighted F1": mean_std_text(group["test_weighted_f1"]),
    "Best epoch": mean_std_text(group["best_epoch"], digits=1),
    "Wall time/run (min)": mean_std_text(
        group["wall_seconds"] / 60, digits=1
    ),
    "Total five-seed time (min)": f"{total_experiment_minutes:.1f}",
    "Peak VRAM (GiB)": mean_std_text(group["peak_vram_gb"], digits=2),
    "Parameters (M)": f"{group['total_parameters'].iloc[0] / 1e6:.2f}",
    "Fixed graph alpha": f"{selected_alpha:.2f}",
    "Base auxiliary weight": f"{selected_base_aux:.2f}",
    "Patience": PATIENCE,
    "Frozen latest architecture": True,
    "Original - shuffled (dev pp)": mean_std_text(
        group["dev_original_minus_shuffled_pp"]
    ),
    "Original - self-loop (dev pp)": mean_std_text(
        group["dev_original_minus_self_loop_pp"]
    ),
    "Graph contribution (dev pp)": mean_std_text(
        group["dev_graph_contribution_pp"]
    ),
}])
final_table.to_csv(ARTIFACT_DIR / "final_results_table.csv", index=False)
print("FINAL TABLE — five seeds retrained from scratch")
display(final_table)

graph_pass = bool(final_summary["graph_evidence_pass"])
dev_gap = BASELINE_DEV_F1 - float(final_summary["dev_macro_f1_mean"])
test_mean = float(group["test_entity_macro_f1"].mean())
collapsed_seeds = group[
    group["dev_graph_off_entity_macro_f1"] < COLLAPSE_THRESHOLD
]["seed"].astype(int).tolist()

if graph_pass and dev_gap < DEV_GAP_LIMIT and test_mean > BASELINE_TEST_MACRO_F1:
    decision_level = "A_supportive_hybrid"
    decision_text = PREDECLARED_DECISION_RULES[
        "rule_A_supportive_hybrid"
    ]["conclusion"]
elif len(collapsed_seeds) >= 2 and not graph_pass:
    decision_level = "B_systematic_base_problem"
    decision_text = PREDECLARED_DECISION_RULES[
        "rule_B_systematic_base_problem"
    ]["conclusion"]
elif dev_gap >= DEV_GAP_LIMIT:
    decision_level = "C_hybrid_not_reliable"
    decision_text = PREDECLARED_DECISION_RULES[
        "rule_C_hybrid_not_reliable"
    ]["conclusion"]
else:
    decision_level = "mixed_inconclusive"
    decision_text = (
        "Five-seed evidence is mixed; report all metrics without a "
        "superiority claim."
    )

test_bootstrap_aggregate = bootstrap_test_df[
    bootstrap_test_df["comparison"].eq(
        "mean_hybrid_across_five_seeds_absolute"
    )
].iloc[0]
decision = {
    "decision_level": decision_level,
    "decision": decision_text,
    "graph_evidence_pass_mean_gt_std": graph_pass,
    "dev_macro_f1_mean": float(final_summary["dev_macro_f1_mean"]),
    "dev_gap_baseline_reference_minus_hybrid": dev_gap,
    "dev_gap_below_0_3pp": bool(dev_gap < DEV_GAP_LIMIT),
    "test_macro_f1_mean": test_mean,
    "test_above_reported_baseline_mean": bool(
        test_mean > BASELINE_TEST_MACRO_F1
    ),
    "test_absolute_bootstrap_ci95_low": float(
        test_bootstrap_aggregate["ci95_low"]
    ),
    "test_absolute_bootstrap_ci95_high": float(
        test_bootstrap_aggregate["ci95_high"]
    ),
    "collapse_threshold": COLLAPSE_THRESHOLD,
    "collapsed_seeds": collapsed_seeds,
    "graph_contribution_pp_mean": float(
        final_summary["graph_contribution_pp_mean"]
    ),
    "graph_contribution_pp_std": float(
        final_summary["graph_contribution_pp_std"]
    ),
    "publication_limitation": (
        "No baseline checkpoint is available. Baseline comparisons use only "
        "previously reported aggregate means and are not paired bootstrap "
        "comparisons. Strict superiority is therefore not claimed."
    ),
}
(ARTIFACT_DIR / "predeclared_decision_outcome.json").write_text(
    json.dumps(decision, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("PREDECLARED DECISION OUTCOME")
display(pd.Series(decision))

comparison_table = pd.DataFrame([
    {
        "Stage": "Baseline Word-CRF (reported 3 seeds)",
        "Dev Entity Macro F1": "0.9256 ± 0.0100",
        "Test Entity Macro F1": "0.9233 ± 0.0207",
        "Graph contribution (dev pp)": "—",
    },
    {
        "Stage": "Latest split-head (previous 3 seeds)",
        "Dev Entity Macro F1": "0.9241 ± 0.0061",
        "Test Entity Macro F1": "0.9460 ± 0.0111",
        "Graph contribution (dev pp)": "1.0972 ± 1.1481",
    },
    {
        "Stage": "Frozen split-head retrained from scratch (5 seeds)",
        "Dev Entity Macro F1": mean_std_text(
            group["dev_entity_macro_f1"]
        ),
        "Test Entity Macro F1": mean_std_text(
            group["test_entity_macro_f1"]
        ),
        "Graph contribution (dev pp)": mean_std_text(
            group["dev_graph_contribution_pp"]
        ),
    },
])
comparison_table.to_csv(
    ARTIFACT_DIR / "before_after_comparison.csv", index=False
)
display(comparison_table)

deploy_row = group.loc[group["dev_entity_macro_f1"].idxmax()]
deploy_checkpoint = ARTIFACT_DIR / f"{MODEL_NAME}_best.pt"
shutil.copy2(deploy_row["checkpoint"], deploy_checkpoint)
(ARTIFACT_DIR / "selected_deploy_run.json").write_text(
    json.dumps(
        deploy_row.to_dict(), ensure_ascii=False, indent=2, default=str
    ),
    encoding="utf-8",
)
print("Deploy checkpoint selected by dev F1:", deploy_checkpoint)


FINAL TABLE — five seeds retrained from scratch


,Model,Seeds,Train runs from scratch,Dev Entity Macro F1,Dev graph-off Macro F1,Test Entity Macro F1,Test Entity Micro F1,Test Weighted F1,Best epoch,Wall time/run (min),Total five-seed time (min),Peak VRAM (GiB),Parameters (M),Fixed graph alpha,Base auxiliary weight,Patience,Frozen latest architecture,Original - shuffled (dev pp),Original - self-loop (dev pp),Graph contribution (dev pp)
0,layoutlmv3_split_head_symbolic_rel_gatv2_fusio...,5,5,0.9247 ± 0.0049,0.9152 ± 0.0058,0.9398 ± 0.0161,0.9764 ± 0.0037,0.9736 ± 0.0032,11.2 ± 1.1,36.2 ± 0.2,192.6,4.40 ± 0.00,132.27,0.50,0.35,6,True,2.0159 ± 0.4385,0.5715 ± 0.5029,0.9464 ± 0.3713


PREDECLARED DECISION OUTCOME


decision_level                                                           A_supportive_hybrid
decision                                   Hybrid has supportive five-seed evidence; pair...
graph_evidence_pass_mean_gt_std                                                         True
dev_macro_f1_mean                                                                   0.924674
dev_gap_baseline_reference_minus_hybrid                                             0.000926
dev_gap_below_0_3pp                                                                     True
test_macro_f1_mean                                                                    0.9398
test_above_reported_baseline_mean                                                       True
test_absolute_bootstrap_ci95_low                                                     0.88965
test_absolute_bootstrap_ci95_high                                                   0.958707
collapse_threshold                                                    

,Stage,Dev Entity Macro F1,Test Entity Macro F1,Graph contribution (dev pp)
0,Baseline Word-CRF (reported 3 seeds),0.9256 ± 0.0100,0.9233 ± 0.0207,—
1,Latest split-head (previous 3 seeds),0.9241 ± 0.0061,0.9460 ± 0.0111,1.0972 ± 1.1481
2,Frozen split-head retrained from scratch (5 se...,0.9247 ± 0.0049,0.9398 ± 0.0161,0.9464 ± 0.3713


Deploy checkpoint selected by dev F1: /kaggle/working/receipt_kie_five_seed_retrain_artifacts/layoutlmv3_split_head_symbolic_rel_gatv2_fusion_crf_best.pt


## 8. Per-Class Performance Analysis

This section aggregates precision, recall, F1, and support for each receipt entity across the five seeds. Sorting classes by mean F1 highlights low-support and difficult fields that may not be visible in aggregate macro and micro scores.


In [13]:
per_class_rows = []
for run_key, split_reports in reports.items():
    model_name, seed_text = run_key.rsplit("_seed", 1)
    test_report = split_reports["test"]
    for label in LABELS[1:]:
        values = test_report.get(label, {})
        per_class_rows.append({
            "model": model_name,
            "seed": int(seed_text),
            "label": label,
            "precision": values.get("precision", 0.0),
            "recall": values.get("recall", 0.0),
            "f1": values.get("f1-score", 0.0),
            "support": values.get("support", 0.0),
        })

per_class_df = pd.DataFrame(per_class_rows)
per_class_df.to_csv(ARTIFACT_DIR / "per_class_results_by_seed.csv", index=False)
per_class_summary = per_class_df.groupby(["model", "label"]).agg(
    f1_mean=("f1", "mean"), f1_std=("f1", "std"),
    precision_mean=("precision", "mean"), recall_mean=("recall", "mean"),
    support=("support", "first"), seeds=("seed", "count"),
).reset_index()
per_class_summary.to_csv(ARTIFACT_DIR / "per_class_summary.csv", index=False)

model_class_summary = per_class_summary[
    per_class_summary["model"] == MODEL_NAME
].sort_values("f1_mean")
print("HYBRID CLASSES — weakest first")
display(model_class_summary)

HYBRID CLASSES — weakest first


,model,label,f1_mean,f1_std,precision_mean,recall_mean,support,seeds
4,layoutlmv3_split_head_symbolic_rel_gatv2_fusio...,S-EMONEY_PAYMENT,0.639444,0.183842,0.550000,0.800000,4.0,5
13,layoutlmv3_split_head_symbolic_rel_gatv2_fusio...,S-OTHER,0.766206,0.047321,0.904494,0.673171,41.0,5
6,layoutlmv3_split_head_symbolic_rel_gatv2_fusio...,S-MENUTYPE_CNT,0.844651,0.022769,0.944762,0.764706,17.0,5
0,layoutlmv3_split_head_symbolic_rel_gatv2_fusio...,S-CARD_PAYMENT,0.958088,0.040253,0.953059,0.964706,51.0,5
8,layoutlmv3_split_head_symbolic_rel_gatv2_fusio...,S-MENU_DISCOUNT_PRICE,0.960434,0.019079,0.948145,0.973333,30.0,5
1,layoutlmv3_split_head_symbolic_rel_gatv2_fusio...,S-CASH,0.962312,0.013413,0.979494,0.945946,148.0,5
5,layoutlmv3_split_head_symbolic_rel_gatv2_fusio...,S-MENUQTY_CNT,0.963588,0.009026,0.940322,0.988060,67.0,5
3,layoutlmv3_split_head_symbolic_rel_gatv2_fusio...,S-DISCOUNT,0.971429,0.063888,1.000000,0.950000,16.0,5
12,layoutlmv3_split_head_symbolic_rel_gatv2_fusio...,S-MENU_UNITPRICE,0.975329,0.008231,0.976889,0.973913,69.0,5
7,layoutlmv3_split_head_symbolic_rel_gatv2_fusio...,S-MENU_CNT,0.980376,0.007030,0.968500,0.992683,246.0,5


## 9. Checkpoints, Metadata, and Reproducibility Artifacts

This section packages the complete hybrid experiment, including:

- The deployment checkpoint selected by development Entity Macro F1.
- Per-seed checkpoints and training histories.
- Final aggregate results and per-seed experiment tables.
- Development ablation results for all four graph conditions.
- Attention, residual, and graph-off diagnostics.
- Per-class reports and document-level predictions.
- Absolute bootstrap results and predeclared decision records.
- Model configuration, seed list, initialization policy, selection protocol, and environment metadata.

The downloadable archive is written to `/kaggle/working/receipt_kie_five_seed_retrain_artifacts_download.zip`.


In [14]:
archive_base = ARTIFACT_DIR.parent / f"{ARTIFACT_DIR.name}_download"
metadata = {
    "task": "CORD receipt key information extraction",
    "architecture": (
        "Frozen latest: LayoutLMv3 + 2-layer symbolic-line Relation-GATv2 + "
        "separate base/fusion classifiers + hidden fusion + Word-CRF"
    ),
    "model_id": CFG.model_id,
    "model_revision": CFG.model_revision,
    "seeds_trained_from_scratch": SEEDS,
    "selected_graph_alpha": selected_alpha,
    "selected_base_aux_weight": selected_base_aux,
    "patience": PATIENCE,
    "bootstrap_repetitions": CFG.bootstrap_repetitions,
    "test_policy": (
        "test evaluated only after all five seeds completed training and "
        "all four dev ablations"
    ),
    "checkpoint_selection": "dev entity macro F1 original mode only",
    "warm_start": False,
    "old_checkpoints_reused": False,
    "architecture_changed_from_latest": False,
    "graph_construction_changed": False,
    "train_runs": len(SEEDS),
    "artifact_dir": str(ARTIFACT_DIR),
    "deploy_checkpoint": str(deploy_checkpoint),
    "download_archive": f"{archive_base}.zip",
}
(ARTIFACT_DIR / "run_metadata.json").write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8"
)
display(pd.Series(metadata))
archive_path = shutil.make_archive(
    str(archive_base), "zip", root_dir=ARTIFACT_DIR
)
print("Download archive:", archive_path)
print("Saved artifacts:")
for path in sorted(ARTIFACT_DIR.iterdir()):
    print(path.name)


task                                          CORD receipt key information extraction
architecture                        Frozen latest: LayoutLMv3 + 2-layer symbolic-l...
model_id                                                    microsoft/layoutlmv3-base
model_revision                                                                   main
seeds_trained_from_scratch                                     [13, 42, 2026, 7, 123]
selected_graph_alpha                                                              0.5
selected_base_aux_weight                                                         0.35
patience                                                                            6
bootstrap_repetitions                                                           10000
test_policy                         test evaluated only after all five seeds compl...
checkpoint_selection                           dev entity macro F1 original mode only
warm_start                                            

Download archive: /kaggle/working/receipt_kie_five_seed_retrain_artifacts_download.zip
Saved artifacts:
before_after_comparison.csv
bootstrap_hybrid_absolute_results.csv
classification_reports.json
dev_ablation_results.csv
dev_configuration_summary.csv
document_predictions.json
epoch_graph_off_diagnostics.csv
experiment_results.csv
final_results_table.csv
latest_frozen_a050_b035_five_seed_retrain_seed_cache.csv
layoutlmv3_split_head_symbolic_rel_gatv2_fusion_crf_best.pt
layoutlmv3_split_head_symbolic_rel_gatv2_fusion_crf_latest_frozen_a050_b035_five_seed_retrain_a050_b035_p6_seed123.pt
layoutlmv3_split_head_symbolic_rel_gatv2_fusion_crf_latest_frozen_a050_b035_five_seed_retrain_a050_b035_p6_seed123_history.csv
layoutlmv3_split_head_symbolic_rel_gatv2_fusion_crf_latest_frozen_a050_b035_five_seed_retrain_a050_b035_p6_seed13.pt
layoutlmv3_split_head_symbolic_rel_gatv2_fusion_crf_latest_frozen_a050_b035_five_seed_retrain_a050_b035_p6_seed13_history.csv
layoutlmv3_split_head_symbolic_rel_ga

## 10. Interpretation and Reporting Scope

The final results should distinguish three claims:

1. **Predictive performance:** report development and test Entity Macro F1 as mean ± sample standard deviation across the five independent seeds.
2. **Graph contribution:** report the original-minus-shuffled, original-minus-self-loop, and original-minus-graph-off differences on the development split.
3. **Uncertainty:** use the saved document-level bootstrap intervals as an absolute uncertainty analysis for the hybrid model.

The baseline comparison uses aggregate metrics rather than matched per-document predictions. Therefore, the experiment supports descriptive comparison with the baseline but does not constitute a paired statistical superiority test. A paired comparison would require baseline predictions or checkpoints evaluated on the same documents and corresponding seeds.

Because the notebook uses reference CORD text and bounding boxes, it evaluates key information extraction with oracle OCR inputs rather than a complete OCR-to-extraction pipeline.
